## **Setup**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from chatGnT.config import CFG, ensure_dirs
import chatGnT.utils as utils
import chatGnT.data.load as load
import chatGnT.data.preprocess as preprocess
import chatGnT.data.tokenize as tokenize
ensure_dirs(CFG)
import time
import math
import torch
import pandas as pd
import numpy as np
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn  # for embedding layer
from torch.utils.data import TensorDataset, DataLoader
import chatGnT.models.transformer as transformer
import chatGnT.models.positional_encoding as positional_encoding
import chatGnT.models.train as train
import chatGnT.models.evaluate as evaluate
import chatGnT.models.predict as predict

/Users/slacksa/miniconda3/envs/chatGnT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Load & Clean Data**

In [4]:
data = load.load_all()
df = data["ingred"]
#TODO: how make units consistent - good opportunity for test functions?


In [11]:
import chatGnT.data.preprocess as preprocess

df_clean = preprocess.clean_recipes(df)
df_clean_filt = preprocess.filter_recipes(df_clean)


## **Make Vocab & Tokenize**

In [ ]:
tokens = tokenize.recipe_to_tokens(df_clean)
print(tokens[0])

In [ ]:
vocab = tokenize.make_vocab(tokens)
print(len(vocab))  # 611 classes to predict
inv_vocab = tokenize.invert_vocab(vocab)
tokens_padded = tokenize.embed_tokens(tokens, vocab)

In [ ]:
# tokens_tensor = [torch.tensor(r) for r in tokens_padded]
tokens_tensor = torch.tensor(tokens_padded, dtype=torch.long)
print(tokens_tensor.shape)  # torch.Size([621, 48])

# Need to have [seq_len, batch_size] for TransformerEncoder
# tokens_tensor = tokens_tensor.transpose(0, 1)
# print(tokens_tensor.shape)  # torch.Size([49, 621])

In [ ]:
print(len(tokens_padded))
type(tokens_padded)
print(tokens_padded[0])

In [ ]:
#TODO: need to do this elsewhere so matches batch size?


# TODO: for now, add padding mask but not a causal mask that block seeing future
pad_id = vocab["<pad>"]   # should be 0
# src_key_padding_mask shape = [batch_size, seq_len]
pad_mask = (tokens_tensor == pad_id).transpose(0, 1)
print(pad_mask.shape)   # [batch_size, seq_len]


## **Get Batches**

In [ ]:
# Not shifting targets since want model to attend to all tokens
inputs = tokens_tensor
targets = tokens_tensor

dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

print(dataloader.batch_size)  # 32
print(len(dataloader))  # 20 = 621 / 32

## **Train Model**

In [ ]:
embed_size=16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformer.TransformerModel(
    ntoken=len(vocab),
    ninp=embed_size,
    nhead=4,
    nhid=256,
    nlayers=2).to(device)
#TODO: investigate error? 

learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_id)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.95)
#TODO: read more about schedular

In [ ]:
best_val_loss = float("inf")
epochs = 12  # number of epochs
best_model = None

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    train.train(model, dataloader, device, pad_id, optimizer, criterion, epoch, 6)
    val_loss = evaluate.evaluate(model, dataloader, device, pad_id, criterion)

    print('-' * 89)
    print(
        f'Epoch {epoch} | Val Loss: {val_loss:.4f} | '
        f'Time {(time.time() - epoch_start_time)} | Val PPL: {math.exp(val_loss):.2f}')
        #TODO: currently not retruning training loss - should I? When look at each?
    print('-' * 89)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model = model

    scheduler.step()  # adjusts learning rate
    #TODO: read more about this?


## **Evaluate with Test Data**

In [ ]:
#TODO: for now, use same data for test but should have separate test set
test_data = dataloader

test_loss = evaluate.evaluate(best_model, test_data, device, pad_id, criterion)
print('=' * 89)
print(f'Test Loss: {test_loss:5.2f} | Test PPL: {math.exp(test_loss):8.2f}')
print('=' * 89)

## **Run with Test User Input**

In [ ]:
#TODO: do need to do general string cleaning... Absinthe vs. absinthe
# <ingred>Absinthe</ingred>': 59
input = "Absinthe"

# tokenize
#TODO: think a different function would be better? Or just need function to handle all
# formats of user input to this format?
# input: df (pd.DataFrame): Must have columns ['amount', 'unit', 'ingred']
#TODO: for now just manually making token
# input_mod = "<ingred>Absinthe</ingred>"
input_mod = ['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Gin</ingred>', '<sep>']

In [ ]:
#TODO: how to embed?
#TODO: consider adding unknown ingredient token for things not in vocab?
predict.predict(best_model, device, pad_id, vocab, inv_vocab, input_mod)




In [ ]:
predict.predict_groups(best_model, device, pad_id, vocab, inv_vocab, input_mod)